# Edge feed patch antenna benchmark

The patch antenna is a ubiquitous antenna type used in modern wireless communication systems. In this notebook, we demonstrate how to simulate a patch antenna using Flexcompute's RF solver and benchmark its performance against a commercial FEM-based full-wave solver. We compare key metrics, such as return loss and gain profile.

<center><img src="./img/edge_feed_patch_antenna_render.png" width=480 /></center>

In [1]:
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import tidy3d as td
import tidy3d.plugins.smatrix as sm
import tidy3d.web as web
import xarray as xr
from tidy3d.plugins.dispersion import FastDispersionFitter
from tidy3d.plugins.microwave import LobeMeasurer

td.config.logging_level = "ERROR"

## Setup

### General Parameters

We will conduct a broadband sweep from 10 MHz to 40 GHz to scan for the resonance(s) of this patch antenna. The target operating frequency of the antenna is around 35.3 GHz.

In [2]:
# Frequency range
f_min, f_max = (0.01e9, 40e9)

# Target operating frequency
f_target = 35.3e9

# Frequency sample points (including f_target)
freqs = np.sort(np.append(np.linspace(f_min, f_max, 301), f_target))

### Material and Structures

Both the substrate and conductor materials are assumed to be lossy and have constant loss parameters over the frequency range. 

In [3]:
# Lossy substrate (rel. epsilon = 2.2, loss tangent = 0.0009)
med_sub = FastDispersionFitter.constant_loss_tangent_model(
    2.2, 0.0009, (f_min, f_max), tolerance_rms=2e-4
)

# Lossy metal (conductivity = 58e6 S/m)
med_metal = td.LossyMetalMedium(conductivity=58, frequency_range=(f_min, f_max))

Output()

The geometry is constructed below. Note that the default length unit is microns. We introduce a scaling factor `mm` for convenience.

In [4]:
# Geometry parameters
mm = 1000
t = 0.02 * mm  # Metal thickness
h = 0.254 * mm  # Substrate thickness

In [5]:
# Create substrate and ground planes
str_sub = td.Structure(
    geometry=td.Box(center=(0, 0, -h / 2), size=(6 * mm, 10 * mm, h)), medium=med_sub
)
str_gnd = td.Structure(
    geometry=td.Box(center=(0, 0, -h - t / 2), size=(6 * mm, 10 * mm, t)), medium=med_metal
)

# Create feed structure
str_feed1 = td.Structure(
    geometry=td.Box.from_bounds(rmin=(2 * mm, -0.31 * mm, 0), rmax=(3 * mm, 0.31 * mm, t)),
    medium=med_metal,
)
str_feed2 = td.Structure(
    geometry=td.Box.from_bounds(rmin=(0.5 * mm, -0.05 * mm, 0), rmax=(2 * mm, 0.05 * mm, t)),
    medium=med_metal,
)

# Create antenna structure
ant_vertices = (
    np.array(
        [
            [-1.3, -2.1],
            [-0.3, -2.1],
            [-0.3, -1.1],
            [0.5, -1.1],
            [0.5, 0.6],
            [-0.19, 0.6],
            [-0.19, -0.62],
            [-1.4, -0.62],
            [-1.4, 0.6],
            [-2.1, 0.6],
            [-2.1, -1.1],
            [-1.3, -1.1],
        ]
    )
    * mm
)
str_ant = td.Structure(
    geometry=td.PolySlab(axis=2, slab_bounds=[0, t], vertices=ant_vertices), medium=med_metal
)

# Full structure list
str_list_full = [str_sub, str_gnd, str_feed1, str_feed2, str_ant]

### Grid and Boundary

We apply the perfectly matched layer (PML) on all external boundaries. As is standard practice for radiation problems, we also include an air buffer region around the antenna. Typically, the thickness of this padding is wavelength/2 on each side. This is so that the PML does not intrude on and distort the near-field. 

In [6]:
# Define simulation size with lambda/2 padding
padding = td.C_0 / f_target / 2
sim_LX = 6 * mm + 2 * padding
sim_LY = 10 * mm + 2 * padding
sim_LZ = 2 * t + h + 2 * padding

In [7]:
# Define PML boundary on all sides
bspec = td.BoundarySpec.all_sides(td.PML())

The grid size in the overall simulation is typically determined by the target wavelength. Thus, in the overall grid specification, we set the maximum grid step size to be wavelength/20. 

That said, it is also important to refine the grid near the metallic structures, as they are responsible for the resonant behavior of the antenna. The `LayerRefinementSpec` serves this purpose. We define a function that creates `LayerRefinementSpec` objects in the top layer (feed structure and antenna) and bottom layer (ground plane) respectively. 

Within each layer, the grid is refined in the normal direction (z) as well as around any metal corners. These are controlled by the `min_steps_along_axis` and `corner_refinement` parameters respectively. 

In [8]:
# Define layer refinement on metallic structures
def create_layer_refinement(structure_list):
    """Create pre-defined layer refinement spec for input structure list"""
    return td.LayerRefinementSpec.from_structures(
        structures=structure_list,
        min_steps_along_axis=2,
        corner_refinement=td.GridRefinement(dl=t / 2, num_cells=2),
    )


lr1 = create_layer_refinement([str_gnd])
lr2 = create_layer_refinement([str_feed1, str_feed2, str_ant])

In [9]:
# Define grid specification
gspec = td.GridSpec.auto(
    wavelength=td.C_0 / f_target,
    min_steps_per_wvl=20,
    layer_refinement_specs=[lr1, lr2],
)

### Excitation

The antenna is fed using a microstrip line with an estimated impedance of 56 ohms using an impedance calculator. We excite the microstrip line with a lumped port of the corresponding impedance, connected to the end of the feed structure.

In [10]:
# Define lumped port excitation
LP1 = sm.LumpedPort(
    center=(3 * mm, 0, -h / 2), size=(0, 0.62 * mm, h), voltage_axis=2, impedance=56, name="LP1"
)

### Monitors

We define two field monitors to visualize the near-field profile at the target resonance frequency. 

In [11]:
# Define near field monitors
mon1 = td.FieldMonitor(
    center=(0, 0, 0), size=(td.inf, 0, td.inf), freqs=[f_target], name="xz plane"
)
mon2 = td.FieldMonitor(
    center=(0, 0, 0), size=(td.inf, td.inf, 0), freqs=[f_target], name="xy plane"
)

surface_mon = td.SurfaceFieldMonitor(
    size=(td.inf, td.inf, td.inf),
    freqs=[f_min, f_target, f_max],
    name="surface",
)

Far-field radiation data is calculated by the `DirectivityMonitor`. We first specify the elevation and azimuthal angular sweep points, followed by the `DirectivityMonitor` that encloses the whole antenna structure. 

In [12]:
# Define elevation and azimuthal angular observation points
# Theta is the elevation angle and defined relative to global +z axis
theta = np.linspace(0, np.pi, 91)
# Phi is the azimuthal angle and defined relative to global +x axis
phi = np.linspace(-np.pi, np.pi, 181)

# The DirectivityMonitor calculates the radiation pattern using a near-to-far-field transformation
mon_radiation = td.DirectivityMonitor(
    center=(0, 0, 0),
    size=(
        0.9 * sim_LX,
        0.9 * sim_LY,
        0.9 * sim_LZ,
    ),  # The monitor should enclose the whole structure of interest
    freqs=[f_target],
    name="radiation",
    phi=phi,
    theta=theta,
    far_field_approx=False,  # Set to False for more accurate computation for slightly higher cost
)

### Simulation and TerminalComponentModeler

The `Simulation` object contains all the information relevant to the simulation defined thus far. 

For broadband simulations that stretch into lower frequencies (<1 GHz), it can be helpful to reduce the `shutoff` threshold to allow for the low frequency content of the time signal to die out. This ensures converged S-parameter values. Be sure to also increase the `run_time` to allow sufficient simulation time to reach the threshold.

In [13]:
# Define simulation object
sim = td.Simulation(
    size=(sim_LX, sim_LY, sim_LZ),
    structures=str_list_full,
    grid_spec=gspec,
    boundary_spec=bspec,
    monitors=[mon1, mon2, surface_mon],
    run_time=3e-9,
    shutoff=1e-7,
    plot_length_units="mm",
)

The `TerminalComponentModeler` (TCM) is a wrapper object that automatically runs a port sweep on the simulation using user-defined `ports` and constructs the full S-parameter matrix. In this case there is only 1 port. The `radiation_monitors` setting is where we include the previously defined `DirectivityMonitor`. 

In [14]:
# Define TerminalComponentModeler
tcm = sm.TerminalComponentModeler(
    simulation=sim,
    ports=[LP1],
    radiation_monitors=[mon_radiation],
    freqs=freqs,
    remove_dc_component=False,  # Set to False when sim is broadband and includes low frequencies (<1 GHz)
)

### Plotting

Before running, we should plot the simulation and check the grid. 

In [15]:
sim_data = web.run(tcm.sim_dict["LP1"], solver_version="surface-mnt-0.0.0")

00:23:13 PDT Created task 'fdtd_2025-10-15_00-23-13' with resource_id           
             'fdve-972cf87f-3250-4568-81b3-34edc345972d' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=357975;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=435628;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\taskId]8;;\]8;id=357975;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\=]8;;\]8;id=367881;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\fdve]8;;\]8;id=357975;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\-972cf87f-325]8;;\
             ]8;id=357975;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\0-4568-81b3-34edc345972d']8;;\.

             Task folder: ]8;id=237319;https://tidy3d.simulation.cloud/folders/93a98b02-262f-4add-a97f-ad569793fe53\'default']8;;\.

Output()

00:23:20 PDT Estimated FlexCredit cost: 0.432. Minimum cost depends on task     
             execution details. Use 'web.real_cost(task_id)' to get the billed  
             FlexCredit cost after a simulation run.

...

Output()

00:24:54 PDT status = success

00:24:56 PDT View simulation result at                                          
             ]8;id=181794;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=75557;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\taskId]8;;\]8;id=181794;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\=]8;;\]8;id=314479;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\fdve]8;;\]8;id=181794;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\-972cf87f-325]8;;\
             ]8;id=181794;https://tidy3d.simulation.cloud/workbench?taskId=fdve-972cf87f-3250-4568-81b3-34edc345972d\0-4568-81b3-34edc345972d']8;;\.

Output()

00:25:02 PDT loading simulation from simulation_data.hdf5

In [16]:
sim_data["surface"].H.real.isel(f=1).norm(dim="axis").plot()

Widget(value='<iframe src="http://localhost:42439/index.html?ui=P_0x7d19b988bbb0_0&reconnect=auto" class="pyvi…